In [50]:
import pandas as pd
import requests
import time
import os

In [51]:
# Load environment variables directly from .env file
from dotenv import load_dotenv
import os

# Automatically finds and loads key-value pairs from .env in workspace
load_dotenv(override=True)

True

In [52]:
# Google Maps API Setup - Automatically loaded from .env
import os
import requests
import json
from dotenv import load_dotenv

# Load .env file
load_dotenv(override=True)

GOOGLE_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY")

if not GOOGLE_API_KEY or GOOGLE_API_KEY == "your_google_maps_api_key_here":
    print("⚠️ GOOGLE_MAPS_API_KEY is not configured in .env file")
    print("   Please update your key in the .env file at the project root.")
else:
    print(f"✅ Google Maps API key loaded from .env (ends with: ...{GOOGLE_API_KEY[-4:]})")

def geocode_place(place_name, api_key=None):
    """Convert place name to lat/lon using Google Geocoding API."""
    api_key = api_key or os.environ.get("GOOGLE_MAPS_API_KEY")
    if not api_key or api_key == "your_google_maps_api_key_here":
        print("❌ No valid Google Maps API key set in .env")
        return None
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": place_name, "key": api_key}
    r = requests.get(url, params=params, timeout=10)
    data = r.json()
    if data["status"] == "OK" and data["results"]:
        loc = data["results"][0]["geometry"]["location"]
        return {
            "place_name": place_name,
            "formatted_address": data["results"][0]["formatted_address"],
            "lat": loc["lat"],
            "lng": loc["lng"],
            "place_id": data["results"][0]["place_id"]
        }
    print(f"❌ Geocoding failed: {data['status']}")
    return None


✅ Google Maps API key loaded from .env (ends with: ...HP3Y)


In [53]:
# Google Maps Directions API - Uses key loaded from .env
import os
import requests
import json
import re
from urllib.parse import unquote
from dotenv import load_dotenv

load_dotenv(override=True)

def get_directions(origin, destination, waypoints=None, api_key=None):
    """Get driving directions using Google Directions API."""
    api_key = api_key or os.environ.get("GOOGLE_MAPS_API_KEY")
    if not api_key or api_key == "your_google_maps_api_key_here":
        print("❌ No valid API key set in .env")
        return None
    
    url = "https://maps.googleapis.com/maps/api/directions/json"
    params = {
        "origin": origin,
        "destination": destination,
        "mode": "driving",
        "key": api_key
    }
    if waypoints:
        params["waypoints"] = "|".join([f"via:{wp}" for wp in waypoints])
    
    r = requests.get(url, params=params, timeout=15)
    data = r.json()
    
    if data["status"] != "OK" or not data["routes"]:
        print(f"❌ Directions failed: {data['status']}")
        if "error_message" in data:
            print(f"   {data['error_message']}")
        return None
    
    route = data["routes"][0]
    legs = []
    total_distance_m = 0
    total_duration_s = 0
    
    for i, leg in enumerate(route["legs"]):
        legs.append({
            "leg_index": i,
            "start_address": leg["start_address"],
            "end_address": leg["end_address"],
            "start_location": leg["start_location"],
            "end_location": leg["end_location"],
            "distance_m": leg["distance"]["value"],
            "distance_text": leg["distance"]["text"],
            "duration_s": leg["duration"]["value"],
            "duration_text": leg["duration"]["text"],
            "steps_count": len(leg["steps"]),
            "steps": [{"start_location": s.get("start_location"), "end_location": s.get("end_location")} for s in leg["steps"]]
        })
        total_distance_m += leg["distance"]["value"]
        total_duration_s += leg["duration"]["value"]
    
    return {
        "total_distance_km": round(total_distance_m / 1000, 2),
        "total_duration_hours": round(total_duration_s / 3600, 2),
        "total_distance_text": f"{total_distance_m/1000:.1f} km",
        "total_duration_text": f"{total_duration_s/3600:.1f} hours",
        "legs": legs,
        "overview_polyline": route["overview_polyline"]["points"]
    }

def parse_google_maps_url(gmaps_url, api_key=None):
    """Parse a long Google Maps URL to extract origin, destination, waypoints."""
    api_key = api_key or os.environ.get("GOOGLE_MAPS_API_KEY")
    if "/maps/dir/" not in gmaps_url:
        print(f"❌ Not a valid Google Maps directions URL")
        return None
    path_part = gmaps_url.split("/maps/dir/")[1].split("/@")[0].split("/data=")[0]
    raw_places = [p for p in path_part.split("/") if p]
    places = [unquote(p.replace("+", " ")) for p in raw_places]
    if len(places) < 2:
        print(f"❌ Need at least origin and destination, got: {places}")
        return None
    return {
        "origin": places[0],
        "destination": places[-1],
        "waypoints": places[1:-1] if len(places) > 2 else None,
        "raw_places": places
    }


In [ ]:
# --- Dynamic Google Maps URL input (Streamlit-like, runs everywhere) ---
# Replaces the hardcoded `long_url = "..."` with a value supplied by the user.
# The widget auto-adapts: st.text_input under Streamlit, input() in a plain
# notebook/CLI. The sample URL is only a fallback default (env-overridable via
# DEFAULT_MAPS_URL). See src/interface/maps_url_interface.py.
import os, sys
if str(os.getcwd()) not in sys.path:
    sys.path.insert(0, str(os.getcwd()))
from src.interface.maps_url_interface import (
    DEFAULT_MAPS_URL,
    acquire_google_maps_url,
    validate_google_maps_url,
    parse_google_maps_url,
)

# Acquire the route URL from the user (backend auto-detected).
long_url = acquire_google_maps_url(
    label="📍 Paste a Google Maps directions URL (origin → destination → waypoints)",
    default=DEFAULT_MAPS_URL,
    help_text="Open maps.google.com → Directions → build a route → Share → Copy link → paste here.",
)

# Always define `directions` so downstream cells never hit a NameError.
directions = None
_ok, _reason = validate_google_maps_url(long_url)
if not _ok:
    print(f"❌ {long_url}\n   → {_reason}")
else:
    print("✅ Valid Google Maps directions URL received.")
    route_info = parse_google_maps_url(long_url)
    print(json.dumps(route_info, indent=2, ensure_ascii=False))
    directions = get_directions(
        origin=route_info["origin"],
        destination=route_info["destination"],
        waypoints=route_info["waypoints"],
    )

print(json.dumps(directions, indent=2, ensure_ascii=False) if directions else "⚠️ No directions available – check the URL / API key.")


In [61]:
# Dynamic Vedur Station Mapper with 15km Spatial Sampling
import os
import requests
import json
import math
import pandas as pd
from typing import Dict, List, Optional, Tuple, Any

VEDUR_BASE_URL = "https://api.vedur.is/weather"

VEDUR_COLUMN_MAP = {
    "station": "station_id",
    "name": "station_name",
    "time": "observation_time_utc",
    "year": "year",
    "month": "month",
    "day": "day",
    "hour": "hour_utc",
    "t": "air_temp_c",
    "tx": "air_temp_max_c",
    "tn": "air_temp_min_c",
    "rh": "relative_humidity_pct",
    "vp": "vapor_pressure_hpa",
    "td": "dew_point_c",
    "f": "wind_speed_avg_ms",
    "fx": "wind_speed_max_ms",
    "fg": "wind_gust_max_ms",
    "fgfx": "gust_factor",
    "d": "wind_dir_deg",
    "d_txt": "wind_dir_cardinal",
    "ps": "station_pressure_hpa",
    "p": "sea_level_pressure_hpa",
    "r": "precipitation_mm",
    "tg": "ground_temp_c",
    "t0": "road_surface_temp_c",
    "count_measurements": "measurement_count",
}

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate Great Circle distance in km between two lat/lon coordinates."""
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2.0) ** 2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon / 2.0) ** 2)
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return R * c

def decode_polyline(polyline_str: str) -> List[Tuple[float, float]]:
    """Decode Google Maps encoded polyline into list of (lat, lng) tuples."""
    index, lat, lng = 0, 0, 0
    coordinates = []
    changes = {'latitude': 0, 'longitude': 0}
    while index < len(polyline_str):
        for unit in ['latitude', 'longitude']:
            shift, result = 0, 0
            while True:
                byte = ord(polyline_str[index]) - 63
                index += 1
                result |= (byte & 0x1f) << shift
                shift += 5
                if not byte >= 0x20:
                    break
            if result & 1:
                changes[unit] = ~(result >> 1)
            else:
                changes[unit] = result >> 1
        lat += changes['latitude']
        lng += changes['longitude']
        coordinates.append((lat / 1e5, lng / 1e5))
    return coordinates

def get_active_vedur_stations(station_types=["sj", "sk"]):
    """Fetch active weather stations from Vedur API (/stations?active=true)."""
    url = f"{VEDUR_BASE_URL}/stations"
    params = {"active": "true"}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        active_stations = [s for s in data if s.get("type") in station_types]
        print(f"✅ Loaded {len(active_stations)} active Vedur stations ({', '.join(station_types)})")
        return active_stations
    except Exception as e:
        print(f"❌ Error fetching stations: {e}")
        return []

def map_google_route_to_vedur_stations(directions_data: Dict[str, Any], max_distance_km: float = 30.0, sample_interval_km: float = 15.0):
    """
    Convert Google Maps route points into nearest active Vedur station IDs using 15km spatial sampling.
    """
    active_stations = get_active_vedur_stations()
    if not active_stations or not directions_data or "legs" not in directions_data:
        return []

    sample_points = []
    legs = directions_data.get("legs", [])
    
    # 1. Start location
    if legs and "start_location" in legs[0]:
        sample_points.append({
            "label": f"Origin: {legs[0].get('start_address', 'Start')}",
            "lat": legs[0]["start_location"]["lat"],
            "lng": legs[0]["start_location"]["lng"]
        })
    
    # 2. Intermediate step locations
    for leg_idx, leg in enumerate(legs):
        steps_data = leg.get("steps", [])
        if isinstance(steps_data, list):
            for step_idx, step in enumerate(steps_data):
                if isinstance(step, dict) and "end_location" in step:
                    end_loc = step["end_location"]
                    if isinstance(end_loc, dict) and "lat" in end_loc and "lng" in end_loc:
                        sample_points.append({
                            "label": f"Leg {leg_idx+1} Step {step_idx+1}",
                            "lat": end_loc["lat"],
                            "lng": end_loc["lng"]
                        })
    
    # 3. End location
    if legs and "end_location" in legs[-1]:
        sample_points.append({
            "label": f"Destination: {legs[-1].get('end_address', 'End')}",
            "lat": legs[-1]["end_location"]["lat"],
            "lng": legs[-1]["end_location"]["lng"]
        })

    # 4. Polyline spatial sampling at 15km intervals along road
    overview_polyline = directions_data.get("overview_polyline")
    if overview_polyline:
        poly_points = decode_polyline(overview_polyline)
        if poly_points:
            accumulated_dist = 0.0
            last_pt = poly_points[0]
            for pt in poly_points[1:]:
                dist = haversine_km(last_pt[0], last_pt[1], pt[0], pt[1])
                accumulated_dist += dist
                if accumulated_dist >= sample_interval_km:
                    sample_points.append({
                        "label": f"Polyline Sample (~{accumulated_dist:.1f}km)",
                        "lat": pt[0],
                        "lng": pt[1]
                    })
                    accumulated_dist = 0.0
                last_pt = pt

    # Match each point to nearest Vedur station
    matched_stations = []
    seen_ids = set()

    for pt in sample_points:
        closest = None
        min_dist = float("inf")
        for st in active_stations:
            st_lat, st_lon = st.get("lat"), st.get("lon")
            if st_lat is None or st_lon is None:
                continue
            dist = haversine_km(pt["lat"], pt["lng"], st_lat, st_lon)
            if dist < min_dist:
                min_dist = dist
                closest = st
        
        if closest and min_dist <= max_distance_km and closest["station"] not in seen_ids:
            seen_ids.add(closest["station"])
            matched_stations.append({
                "station_id": closest["station"],
                "station_name": closest.get("name"),
                "lat": closest.get("lat"),
                "lon": closest.get("lon"),
                "distance_km": round(min_dist, 2),
                "matched_point": pt["label"]
            })

    return matched_stations

def fetch_weather_for_station_ids(station_ids: List[int]):
    """Fetch latest hourly observation data from Vedur API for matched station IDs."""
    url = f"{VEDUR_BASE_URL}/observations/aws/hour/latest"
    all_obs = []
    for st_id in station_ids:
        try:
            r = requests.get(url, params={"station_id": st_id, "parameters": "all"}, timeout=10)
            r.raise_for_status()
            data = r.json()
            if data:
                all_obs.extend(data)
                print(f"✅ Station {st_id} ({data[0].get('name', '')}): {len(data)} record(s)")
        except Exception as e:
            print(f"❌ Station {st_id}: Error {e}")
    return pd.DataFrame(all_obs)


In [62]:
# Vedur API column name -> full description mapping
VEDUR_COLUMN_MAP = {
    # Station metadata
    "station": "station_id",
    "name": "station_name",
    
    # Time dimensions
    "time": "observation_time_utc",
    "year": "year",
    "month": "month",
    "day": "day",
    "hour": "hour_utc",  # 1-24, where 24 = midnight
    
    # Temperature (Celsius)
    "t": "air_temp_c",           # Air temperature at observation time
    "tx": "air_temp_max_c",      # Maximum air temperature since last observation
    "tn": "air_temp_min_c",      # Minimum air temperature since last observation
    
    # Humidity & moisture
    "rh": "relative_humidity_pct",  # Relative humidity (%)
    "vp": "vapor_pressure_hpa",     # Vapor pressure (hPa)
    "td": "dew_point_c",            # Dew point temperature (C)
    
    # Wind
    "f": "wind_speed_avg_ms",       # Average wind speed (m/s) over 10-min period
    "fx": "wind_speed_max_ms",      # Maximum 10-min average wind speed (m/s)
    "fg": "wind_gust_max_ms",       # Maximum wind gust (m/s)
    "fgfx": "gust_factor",          # Ratio of max gust to max 10-min wind (fg/fx)
    "d": "wind_dir_deg",            # Wind direction (degrees, 0-360)
    "d_txt": "wind_dir_cardinal",   # Wind direction (cardinal: N, NE, E, etc.)
    "dsdev": "wind_dir_std_dev",    # Standard deviation of wind direction (degrees)
    
    # Pressure
    "ps": "station_pressure_hpa",   # Station level pressure (hPa)
    "p": "sea_level_pressure_hpa",  # Sea level pressure (hPa)
    
    # Precipitation
    "r": "precipitation_mm",        # Precipitation since last observation (mm)
    
    # Ground temperature
    "tg": "ground_temp_c",          # Ground temperature (C)
    "tgn": "ground_temp_min_c",     # Minimum ground temperature (C)
    
    # Road surface temperature (for road stations)
    "t0": "road_surface_temp_c",    # Road surface temperature (C)
    "t0x": "road_surface_temp_max_c",
    "t0n": "road_surface_temp_min_c",
    
    # Turf/grass temperature at various depths
    "tug5": "turf_temp_5cm_c",
    "tug10": "turf_temp_10cm_c",
    "tug15": "turf_temp_15cm_c",
    "tug20": "turf_temp_20cm_c",
    "tug50": "turf_temp_50cm_c",
    "tug100": "turf_temp_100cm_c",
    
    # Radiation
    "radgl": "global_radiation_wm2",       # Global radiation (W/m²)
    "radglx": "global_radiation_max_wm2",
    "radsc": "shortwave_radiation_wm2",    # Shortwave radiation (W/m²)
    "radscx": "shortwave_radiation_max_wm2",
    "radsws": "sunshine_duration_s",       # Sunshine duration (seconds)
    "radswsx": "sunshine_duration_max_s",
    "radlwi": "longwave_in_radiation_wm2", # Longwave incoming radiation (W/m²)
    "radlwix": "longwave_in_radiation_max_wm2",
    "radlws": "longwave_out_radiation_wm2", # Longwave outgoing radiation (W/m²)
    "radlwsx": "longwave_out_radiation_max_wm2",
    "raduv": "uv_radiation_wm2",           # UV radiation (W/m²)
    "raduvx": "uv_radiation_max_wm2",
    
    # Other
    "rsun": "sunshine_duration_min",       # Sunshine duration (minutes)
    "ts": "snow_depth_cm",                 # Snow depth (cm)
    "sal": "salinity_psu",                 # Salinity (PSU) - for marine stations
    "count_measurements": "measurement_count",  # Number of measurements in period
}

# Print nicely formatted
for short, full in VEDUR_COLUMN_MAP.items():
    print(f"  {short:>15} -> {full}")

          station -> station_id
             name -> station_name
             time -> observation_time_utc
             year -> year
            month -> month
              day -> day
             hour -> hour_utc
                t -> air_temp_c
               tx -> air_temp_max_c
               tn -> air_temp_min_c
               rh -> relative_humidity_pct
               vp -> vapor_pressure_hpa
               td -> dew_point_c
                f -> wind_speed_avg_ms
               fx -> wind_speed_max_ms
               fg -> wind_gust_max_ms
             fgfx -> gust_factor
                d -> wind_dir_deg
            d_txt -> wind_dir_cardinal
            dsdev -> wind_dir_std_dev
               ps -> station_pressure_hpa
                p -> sea_level_pressure_hpa
                r -> precipitation_mm
               tg -> ground_temp_c
              tgn -> ground_temp_min_c
               t0 -> road_surface_temp_c
              t0x -> road_surface_temp_max_c
              t0n ->

In [ ]:
# ── Fuel Price Fetcher ──────────────────────────────────────────────────────
# Fetches live ISK prices from Gasvaktin CDN with local fallback cache.
# Source: https://github.com/gasvaktin/gasvaktin

import json
import os
import requests
from pathlib import Path
from typing import List, Dict, Optional, Tuple

GASVAKTIN_URL = "https://raw.githubusercontent.com/gasvaktin/gasvaktin/master/vaktin/gas.json"
FUEL_CACHE_FILE = Path("data/gas_price_cache.json")
_STATIONS_CACHE = [None]  # module-level memoisation


def _load_gasvaktin_stations() -> List[Dict]:
    """Load stations from Gasvaktin CDN, falling back to local cache."""
    if _STATIONS_CACHE[0] is not None:
        return _STATIONS_CACHE[0]
    try:
        print("⛽ Fetching live fuel prices from Gasvaktin CDN...")
        r = requests.get(GASVAKTIN_URL, timeout=10)
        r.raise_for_status()
        data = r.json()
        stations = data.get("stations", [])
        if stations:
            os.makedirs(FUEL_CACHE_FILE.parent, exist_ok=True)
            with open(FUEL_CACHE_FILE, "w", encoding="utf-8") as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            print(f"✅ Loaded {len(stations)} fuel stations (cache updated)")
            _STATIONS_CACHE[0] = stations
            return stations
    except Exception as e:
        print(f"⚠️  CDN fetch failed ({e}), trying local cache...")

    if FUEL_CACHE_FILE.exists():
        with open(FUEL_CACHE_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        stations = data.get("stations", [])
        print(f"✅ Loaded {len(stations)} stations from local cache")
        _STATIONS_CACHE[0] = stations
        return stations

    print("❌ No fuel data available online or in cache")
    return []


def get_nearest_fuel_stations(
    lat: float,
    lon: float,
    max_distance_km: float = 50.0,
    limit: int = 10,
) -> List[Dict]:
    """Find nearest fuel stations to a coordinate, sorted by distance."""
    stations = _load_gasvaktin_stations()
    results = []
    for s in stations:
        geo = s.get("geo", {})
        slat, slon = geo.get("lat"), geo.get("lon")
        if slat is None or slon is None:
            continue
        d = haversine_km(lat, lon, slat, slon)
        if d <= max_distance_km:
            results.append({**s, "distance_km": round(d, 2)})
    results.sort(key=lambda x: x["distance_km"])
    return results[:limit]


def get_fuel_price_at_route(
    waypoints: List[Tuple[float, float]],
    fuel_type: str = "bensin95",
    max_distance_km: float = 30.0,
) -> List[Dict]:
    """Find unique fuel stations near a route, sorted by effective price."""
    seen_keys, all_stations = set(), []
    for lat, lon in waypoints:
        for s in get_nearest_fuel_stations(lat, lon, max_distance_km, limit=5):
            key = s.get("key", "")
            if key not in seen_keys:
                seen_keys.add(key)
                base = s.get(fuel_type)
                disc = s.get(f"{fuel_type}_discount")
                effective = disc if disc and disc > 0 else base
                all_stations.append({
                    "station_name": s.get("name"),
                    "company": s.get("company"),
                    "price_isk": effective,
                    "regular_price_isk": base,
                    "discount_price_isk": disc,
                    "distance_km": s.get("distance_km"),
                    "lat": s.get("geo", {}).get("lat"),
                    "lon": s.get("geo", {}).get("lon"),
                    "key": key,
                })
    all_stations.sort(key=lambda x: x.get("price_isk") or float("inf"))
    return all_stations


In [ ]:
# Route Weather Ingestion -> Parquet Export -> DuckDB Analytics
import os
import duckdb
import pandas as pd

# 1. Resolve Route Data (requires Google Maps API key)
route_data = None
if "directions" in locals() and isinstance(directions, dict):
    route_data = directions
elif "route_info" in locals() and isinstance(route_info, dict):
    route_data = get_directions(route_info["origin"], route_info["destination"], route_info.get("waypoints"))

if not route_data:
    print("❌ No route data available. Google Maps API key is not set or directions call failed.")
    print("   Please provide a valid GOOGLE_MAPS_API_KEY in your .env file and re-run cells 3–4.")
    raise SystemExit("Pipeline halted: route data required to continue.")

# 2. Map Route to Weather Stations (15km spatial sampling)
matched_stations = map_google_route_to_vedur_stations(route_data, sample_interval_km=15.0)
print(f"✅ Matched {len(matched_stations)} weather stations along the route.")

if not matched_stations:
    print("❌ No weather stations matched along the route. Cannot continue.")
    raise SystemExit("Pipeline halted: no matched weather stations.")

# 3. Fetch Live Weather Data for ALL Matched Stations
station_ids = [m["station_id"] for m in matched_stations]
df_weather_raw = fetch_weather_for_station_ids(station_ids)

if df_weather_raw.empty:
    print("❌ Vedur API returned no data. Cannot continue.")
    raise SystemExit("Pipeline halted: no weather data returned.")

# 4. Enrich & Rename Data for DuckDB Pipeline
df_weather_renamed = df_weather_raw.rename(columns=VEDUR_COLUMN_MAP)

df_stations_meta = pd.DataFrame(matched_stations)
df_route_telemetry = pd.merge(
    df_weather_renamed,
    df_stations_meta[["station_id", "lat", "lon", "distance_km", "matched_point"]],
    on="station_id",
    how="left"
)

if "waypoint_id" not in df_route_telemetry.columns:
    df_route_telemetry["waypoint_id"] = range(1, len(df_route_telemetry) + 1)
if "stop_name" not in df_route_telemetry.columns:
    df_route_telemetry["stop_name"] = df_route_telemetry["station_name"]
if "region" not in df_route_telemetry.columns:
    df_route_telemetry["region"] = "Data not available, please try later"
if "latitude" not in df_route_telemetry.columns:
    df_route_telemetry["latitude"] = df_route_telemetry["lat"]
if "longitude" not in df_route_telemetry.columns:
    df_route_telemetry["longitude"] = df_route_telemetry["lon"]
if "timestamp" not in df_route_telemetry.columns:
    df_route_telemetry["timestamp"] = df_route_telemetry["observation_time_utc"]
if "wind_speed_ms" not in df_route_telemetry.columns:
    df_route_telemetry["wind_speed_ms"] = df_route_telemetry.get("wind_speed_avg_ms")
if "wind_gust_ms" not in df_route_telemetry.columns:
    df_route_telemetry["wind_gust_ms"] = df_route_telemetry.get("wind_gust_max_ms")
if "temperature_c" not in df_route_telemetry.columns:
    df_route_telemetry["temperature_c"] = df_route_telemetry.get("air_temp_c")
if "precipitation_mm" not in df_route_telemetry.columns:
    df_route_telemetry["precipitation_mm"] = df_route_telemetry.get("precipitation_mm")
if "road_status" not in df_route_telemetry.columns:
    df_route_telemetry["road_status"] = df_route_telemetry["wind_gust_ms"].apply(
        lambda g: "Impassable" if pd.notna(g) and g >= 20 else (
            "Caution Advised" if pd.notna(g) and g >= 14 else "Open"
        )
    )

# 5. Save Weather Telemetry to Parquet & CSV First
parquet_path = "data/iceland_raw_telemetry.parquet"
csv_path = "data/iceland_raw_telemetry.csv"

# Ensure precipitation_mm is a float in pandas before exporting
df_route_telemetry["precipitation_mm"] = pd.to_numeric(df_route_telemetry["precipitation_mm"], errors="coerce")

df_route_telemetry.to_parquet(parquet_path, index=False)
df_route_telemetry.to_csv(csv_path, index=False)
print(f"\n💾 Saved all {len(df_route_telemetry)} station records to {parquet_path} and {csv_path}")

# 6. DuckDB Query: Weather Hazards
conn = duckdb.connect()
df_duckdb = conn.execute("""
    SELECT
    waypoint_id,
    stop_name,
    latitude,
    longitude,
    wind_speed_ms,
    wind_gust_ms,
    temperature_c,
    precipitation_mm,
    road_status,
    CASE
        WHEN wind_gust_ms >= 18 THEN 'CRITICAL: Rollover Risk'
        WHEN wind_gust_ms >= 14 THEN 'HIGH: Strong Crosswinds'
        WHEN wind_gust_ms >= 10 THEN 'MODERATE: Caution'
        ELSE 'CLEAR: Safe Driving'
    END AS hazard_alt,
    -- Last‑value fill (LOCF) for weather fields
    LAST_VALUE(wind_speed_ms IGNORE NULLS) OVER (
        PARTITION BY station_id
        ORDER BY observation_time_utc
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS wind_speed_ms_filled,
    LAST_VALUE(wind_gust_ms IGNORE NULLS) OVER (
        PARTITION BY station_id
        ORDER BY observation_time_utc
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS wind_gust_ms_filled,
    LAST_VALUE(temperature_c IGNORE NULLS) OVER (
        PARTITION BY station_id
        ORDER BY observation_time_utc
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS temperature_c_filled,
    COALESCE(LAST_VALUE(precipitation_mm IGNORE NULLS) OVER (
        PARTITION BY station_id
        ORDER BY observation_timeS_utc
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 0.0) AS precipitation_mm_filled
FROM read_parquet('data/iceland_raw_telemetry.parquet')
ORDER BY waypoint_id, observation_time_utc
""").df()

print("\n--- DUCKDB QUERY RESULTS (FOR STREAMLIT) ---")
print(df_duckdb.to_string(index=False))S

✅ Loaded 309 active Vedur stations (sj, sk)
✅ Matched 22 weather stations along the route.
✅ Station 1469 (Reykjavík Hljómskálagarður): 1 record(s)
✅ Station 1475 (Reykjavík - Bústaðavegur): 1 record(s)
✅ Station 1482 (Reykjavík Víðidalur): 1 record(s)
✅ Station 1493 (Ölkelduháls): 1 record(s)
✅ Station 31399 (Ingólfsfjall): 1 record(s)
✅ Station 6300 (Selfoss): 1 record(s)
✅ Station 6315 (Hella): 1 record(s)
✅ Station 6272 (Kirkjubæjarklaustur - Stjórnarsandur): 1 record(s)
✅ Station 6499 (Skaftafell): 1 record(s)
✅ Station 1481 (Hólmsheiði): 1 record(s)
✅ Station 1490 (Hellisskarð): 1 record(s)
✅ Station 36308 (Þjórsárbrú): 1 record(s)
✅ Station 6222 (Sámsstaðir): 1 record(s)
✅ Station 36127 (Hvammur): 1 record(s)
✅ Station 6134 (Önundarhorn): 1 record(s)
✅ Station 6045 (Vatnsskarðshólar): 1 record(s)
✅ Station 6049 (Vík í Mýrdal): 1 record(s)
✅ Station 36156 (Mýrdalssandur): 1 record(s)
❌ Station 26064: Error 404 Client Error: Not Found for url: https://api.vedur.is/weather/observat

In [75]:
# Fuel Prices Ingestion -> Parquet Export -> DuckDB Analytics
import os
import duckdb
import pandas as pd

# 1. Fetch Live Fuel Prices from Gasvaktin CDN
print("\n⛽ Fetching live fuel prices along route...")
route_waypoints = [
    (row["latitude"], row["longitude"])
    for _, row in df_route_telemetry[["latitude", "longitude"]].drop_duplicates().iterrows()
]
fuel_stations = get_fuel_price_at_route(route_waypoints, fuel_type="bensin95", max_distance_km=40.0)

if fuel_stations:
    df_fuel = pd.DataFrame(fuel_stations)

    # 2a. Enrich telemetry with nearest station's price per waypoint
    def _nearest_fuel(lat, lon):
        df_fuel["_d"] = df_fuel.apply(
            lambda r: haversine_km(lat, lon, r["lat"], r["lon"]) if pd.notna(r["lat"]) else float("inf"),
            axis=1,
        )
        nearest = df_fuel.loc[df_fuel["_d"].idxmin()]
        return nearest.get("price_isk"), nearest.get("station_name", "Unknown")

    df_route_telemetry[["fuel_price_isk", "fuel_brand"]] = df_route_telemetry.apply(
        lambda row: pd.Series(_nearest_fuel(row["latitude"], row["longitude"])),
        axis=1,
    )

    # 2b. Save unique fuel stations to their own parquet
    os.makedirs("data", exist_ok=True)
    fuel_parquet_path = "data/iceland_fuel_stations.parquet"
    df_fuel.drop(columns=["_d"], errors="ignore").to_parquet(fuel_parquet_path, index=False)
    print(f"⛽ Saved {len(df_fuel)} fuel stations to {fuel_parquet_path}")

    # 2c. DuckDB query: closest unique stations along the route
    conn_fuel = duckdb.connect()
    df_fuel_display = conn_fuel.execute("""
        SELECT
            ROW_NUMBER() OVER (ORDER BY price_isk ASC) AS rank,
            station_name,
            company,
            ROUND(COALESCE(discount_price_isk, 0), 1) AS discount_price_isk,
            LAST_VALUE(price_isk IGNORE NULLS) OVER (
            ORDER BY ROUND(distance_km,1) ASC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS price_isk,
            LAST_VALUE(regular_price_isk IGNORE NULLS) OVER (
            ORDER BY ROUND(distance_km,1) ASC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS regular_price_isk,
            ROUND(distance_km, 1)        AS distance_to_route_km,
            ROUND(lat, 5)                AS lat,
            ROUND(lon, 5)                AS lon
        FROM read_parquet('data/iceland_fuel_stations.parquet')
        ORDER BY distance_to_route_km ASC
    """).df()

    print("\n--- DUCKDB QUERY RESULTS (FOR STREAMLIT) ---")
    print(df_fuel_display.to_string(index=False))

else:
    print("⚠️  No fuel stations found along route")
    df_route_telemetry["fuel_price_isk"] = "Data not available, please try later"
    df_route_telemetry["fuel_brand"] = "Data not available, please try later"



⛽ Fetching live fuel prices along route...
⛽ Saved 44 fuel stations to data/iceland_fuel_stations.parquet

--- DUCKDB QUERY RESULTS (FOR STREAMLIT) ---
 rank             station_name     company  discount_price_isk  price_isk  regular_price_isk  distance_to_route_km      lat       lon
   39               Hringbraut          N1               220.9      220.9              230.9                   0.4 64.13882 -21.93820
   23               Birkimelur       Orkan               214.1      214.1              226.1                   0.5 64.14213 -21.95296
   15               Miklabraut       Orkan               214.1      214.1              226.1                   0.7 64.13266 -21.89310
   14 Miklabraut við Kringluna       Orkan               214.1      214.1              226.1                   0.7 64.13205 -21.89338
    1               Skógarhlíð       Orkan                 0.0      200.1              200.1                   0.9 64.13190 -21.91713
   10            Norðlingaholt          N1 